## Objectives 
**Build leakage-safe classification and regression pipelines. 
Compare models using cross-validation and tune selected hyperparameters. 
Interpret fold variability and justify a final model.** 

## Experiment A — classification 

**Load the classification dataset. Exclude ticket_id and separate escalated as the target. Create a stratified train/test split and hold out the test set.** 

In [31]:
import numpy as np, pandas as pd 
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV 
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler 
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor 
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error
clf_df = pd.read_csv("data/classification_escalation.csv") 
reg_df = pd.read_csv("data/resolution_time_regression.csv") 
display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [32]:
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])

y_clf = clf_df["escalated"]

display(X_clf.head())
display(y_clf.head())

,complexity_score,team_experience_years,priority,channel,customer_tenure_months
0,2.0,15.0,Medium,Email,61.818896
1,9.0,6.0,Critical,Chat,110.024317
2,10.0,8.0,High,Phone,36.311244
3,2.0,1.0,Medium,Phone,85.127855
4,8.0,14.0,Medium,Chat,18.225059


0    0
1    0
2    0
3    0
4    0
Name: escalated, dtype: int64

We separate the dataset into features (X_clf), which the model uses to predict, and the target (y_clf), which is what it predicts.

We remove ticket_id because it is only an identifier, and remove escalated from the features because it is the target. This prevents the model from seeing the answer during training (target leakage).

In [33]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.25,
    random_state=42,
    stratify=y_clf
)

We split the data into 75% training and 25% testing. The model learns from the training set; the test set is held aside to evaluate how well it predicts unseen tickets.

stratify=y_clf keeps the proportion of escalated and non-escalated tickets similar in both sets. This is important because escalated tickets are the minority class.

**Build a pipeline with median imputation and scaling for numeric columns, and most frequent imputation and one-hot encoding for categorical columns.**

In [34]:
numeric_features = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months"
]

categorical_features = [
    "priority",
    "channel"
]

In [35]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [36]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [37]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

The dataset has two kinds of input: numbers and categories. We handle them separately because they need different preparation.
- For numerical columns, we fill missing values with the median (the middle value) and scale the numbers so columns with larger values do not dominate just because of their size.
- For categorical columns, we fill missing values with the most common value and use one-hot encoding to represent each category as a numeric column.

**Compare logistic regression, k-NN and decision tree using identical 5-fold StratifiedKFold CV. Use F1 as primary metric; report mean and standard deviation.** 

In [38]:
clf_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

**n_splits=5** divides the training data into 5 parts. Each run trains on 4 parts and evaluates on the remaining part. This repeats until each part has been used for evaluation. **shuffle=True** shuffles the data before dividing it into parts. **random_state=42** makes the shuffle repeatable, so we get the same folds when we run the code again.
**StratifiedKFold** keeps the proportion of escalated and non-escalated tickets similar in each part. This matters because escalated tickets are less common.

In [39]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier())
])

tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

At this stage, we create three complete model pipelines to compare: Logistic Regression, K-Nearest Neighbors, and Decision Tree.

In [40]:
classification_models = {
    "Logistic Regression": logistic_pipeline,
    "KNN": knn_pipeline,
    "Decision Tree": tree_pipeline
}

classification_results = []

for name, model in classification_models.items():

    scores = cross_validate(
        model,
        X_train_clf,
        y_train_clf,
        cv=clf_cv,
        scoring="f1"
    )

    classification_results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

classification_results_df = pd.DataFrame(classification_results)

display(classification_results_df.sort_values(
    by="Mean F1",
    ascending=False
))

,Model,Mean F1,Std F1
2,Decision Tree,0.401449,0.104140
0,Logistic Regression,0.382200,0.053838
1,KNN,0.309444,0.073750


the three models are compared using the same 5-fold cross-validation and F1 score so it’s a fair comparison.

The Decision Tree comes out on top with a mean F1 of 0.401449, followed by Logistic Regression at 0.3822, then KNN at 0.30944. The standard deviation tells us how much each model’s score changed across the folds. Logistic Regression was the most consistent, while the Decision Tree’s scores varied more.

**Tune at least one candidate using GridSearchCV or RandomizedSearchCV. Evaluate the selected pipeline on the test set once and discuss false positives and false negatives.** 

In [41]:
param_grid_clf = {
    "model__max_depth": [None, 3, 5, 7, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10]
}

we used the nmodel decision tree to tune as it has highest f1 score

Hyperparameters are settings we choose before training, rather than values the model learns from the data.

We’re using Grid Search, which tests every combination of the settings we list. It’s a good fit here because we only have a few values to try.
- max_depth sets how deep the tree is allowed to grow. A smaller depth keeps the tree simpler. None means there’s no set depth limit.
- min_samples_split sets how many samples a node must have before the tree can split it into smaller groups. A higher value makes the tree less likely to create lots of small branches.
We’re tuning these because they control how complex the tree gets, which can affect how well it predicts new tickets. Grid Search will compare every combinations using F1 score and help us pick the best settings.

In [42]:
grid_search_clf = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid_clf,
    scoring="f1",
    cv=clf_cv
)

In [43]:
grid_search_clf = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid_clf,
    scoring="f1",
    cv=clf_cv
)

In [46]:
grid_search_clf.fit(X_train_clf, y_train_clf)

print("Best parameters:", grid_search_clf.best_params_)
print("Best CV F1:", grid_search_clf.best_score_)

Best parameters: {'model__max_depth': 10, 'model__min_samples_split': 2}
Best CV F1: 0.4267090442734471


Grid Search run on the Decision Tree pipeline. It tries each combination of the hyperparameter values using the same 5-fold cross-validation and compares them by F1 score.

The best settings were max_depth=10 and min_samples_split=2, with a mean CV F1 of 0.427. That’s higher than the Decision Tree’s earlier score of 0.401, so tuning improved its cross-validation result.

In [45]:
best_clf_model = grid_search_clf.best_estimator_

y_pred_clf = best_clf_model.predict(X_test_clf)

In [17]:
cm = confusion_matrix(
    y_test_clf,
    y_pred_clf,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

test_f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0

print("Test F1:", test_f1)
print("\nConfusion Matrix:")
print(cm)

print("\nTrue Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

Test F1: 0.3469387755102041

Confusion Matrix:
[[144  33]
 [ 31  17]]

True Negatives: 144
False Positives: 33
False Negatives: 31
True Positives: 17


THe above checks the tuned Decision Tree on the test set which it hasn’t seen during training or tuning. This gives us a more realistic check of how it may perform on new tickets.
The test F1 is 0.347, lower than its best cross-validation F1 of 0.427. So, the model did better during tuning than it did on this unseen data.

**144 true negatives:** correctly predicted tickets that were not escalated.**33 false positives:** predicted escalation, but the ticket wasn’t escalated.
**31 false negatives:** predicted not escaleted but the ticket was escalated.
**17 true positives:** correctly identified escalated tickets.

## Experiment B — regression

**Predict resolution_time_hours. Exclude ticket_id and use a train/test split.**

In [18]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])

y_reg = reg_df["resolution_time_hours"]

display(X_reg.head())
display(y_reg.head())

,complexity_score,team_experience_years,priority,channel,customer_tenure_months
0,7.0,10.0,Medium,Chat,32.638377
1,4.0,15.0,Medium,Chat,45.273760
2,4.0,13.0,Low,Portal,44.878323
3,1.0,10.0,High,Phone,40.095034
4,3.0,15.0,Low,Portal,47.451134


0    8.38724
1    0.50000
2    0.50000
3    0.50000
4    0.50000
Name: resolution_time_hours, dtype: float64

In [19]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42
)

print("Training features:", X_train_reg.shape)
print("Testing features:", X_test_reg.shape)
print("Training target:", y_train_reg.shape)
print("Testing target:", y_test_reg.shape)

Training features: (600, 5)
Testing features: (200, 5)
Training target: (600,)
Testing target: (200,)


**Compare linear regression, ridge regression and decision tree regression using 5-fold CV and MAE.**

In [20]:
reg_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [21]:
regression_models = {
    "Linear Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]),
    "Ridge Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge())
    ]),
    "Decision Tree Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeRegressor(random_state=42))
    ])
}

regression_results = []

for name, model in regression_models.items():
    scores = cross_validate(
        model,
        X_train_reg,
        y_train_reg,
        cv=reg_cv,
        scoring="neg_mean_absolute_error"
    )

    mae_scores = -scores["test_score"]

    regression_results.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

regression_results_df = pd.DataFrame(regression_results)

display(
    regression_results_df.sort_values(
        by="Mean MAE",
        ascending=True
    )
)

,Model,Mean MAE,Std MAE
0,Linear Regression,2.034322,0.095611
1,Ridge Regression,2.034609,0.094634
2,Decision Tree Regression,3.144527,0.340650


compared three models for predicting resolution time. same 5-fold cross-validation setup used, but measured performance with MAE(the average size of the model’s prediction errors, in hours). Lower MAE is better.

Linear Regression and Ridge Regression perform almost identically, with mean MAEs of about 2.03 hours. Decision Tree Regression has a higher mean MAE of 3.14 hours, so its predictions are less accurate on average.

The standard deviations show how much the MAE changed across folds. The scores were fairly consistent for Linear and Ridge Regression, while the Decision Tree varied more. Next, we’ll tune Ridge Regression to see if we can improve its result.

**Tune ridge alpha or tree depth. Report mean and standard deviation of CV MAE, then 
evaluate the selected model on the test set once.**

In [49]:
param_grid_reg = {
    "model__fit_intercept": [True, False]
}

grid_search_reg = GridSearchCV(
    estimator=regression_models["Linear Regression"],
    param_grid=param_grid_reg,
    scoring="neg_mean_absolute_error",
    cv=reg_cv
)

grid_search_reg.fit(X_train_reg, y_train_reg)

print("Best parameters:", grid_search_reg.best_params_)
print("Best CV MAE:", -grid_search_reg.best_score_)
print("Std CV MAE:", grid_search_reg.cv_results_["std_test_score"][best_index])

Best parameters: {'model__fit_intercept': True}
Best CV MAE: 2.034322467440629
Std CV MAE: 0.0956114916203263


fit_intercept is a hyperparameter that decides whether Linear Regression should learn an intercept—the model’s starting value when all input features are zero.
- fit_intercept=True: the model learns that starting value from the training data.
- fit_intercept=False: the model is forced to use zero as the starting value.
Tuning didn’t improve Linear Regression. Grid Search selected fit_intercept=True, the model’s default setting, so the mean CV MAE stayed the same. This suggests the original setting was already the better choice of the two tested.

In [52]:
y_pred = grid_search_reg.best_estimator_.predict(X_test_reg)

print("Test MAE:", mean_absolute_error(y_test_reg, y_pred))

Test MAE: 2.1383137183979564


The final test MAE was 2.1383 hours, meaning the model's predictions differed from the actual resolution times by about 2.14 hours on average. This is slightly higher than the CV MAE of 2.0343 hours but the difference is small, suggesting similar performance on unseen data.

**Explain outliers and why MAE and RMSE may prefer different models.**

In [50]:
Q1 = y_train_reg.quantile(0.25)
Q3 = y_train_reg.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = y_train_reg[
    (y_train_reg < lower_bound) |
    (y_train_reg > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential outliers:", len(outliers))
print("Outlier values:")
display(outliers.sort_values())

Q1: 2.588961147550939
Q3: 8.125645950524813
IQR: 5.536684802973874
Lower bound: -5.716066056909873
Upper bound: 16.430673154985627
Potential outliers: 8
Outlier values:


19     17.981195
113    18.267015
298    23.183383
51     23.929861
573    24.017656
502    24.100668
188    26.135421
193    26.615113
Name: resolution_time_hours, dtype: float64

In [51]:
regression_metric_results = []

for name, model in regression_models.items():
    scores = cross_validate(
        model,
        X_train_reg,
        y_train_reg,
        cv=reg_cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error"
        }
    )

    fold_mae = -scores["test_mae"]
    fold_rmse = np.sqrt(-scores["test_mse"])

    regression_metric_results.append({
        "Model": name,
        "Mean MAE": fold_mae.mean(),
        "Std MAE": fold_mae.std(),
        "Mean RMSE": fold_rmse.mean(),
        "Std RMSE": fold_rmse.std()
    })

regression_metric_results_df = pd.DataFrame(regression_metric_results)

display(
    regression_metric_results_df.sort_values(
        by="Mean MAE",
        ascending=True
    )
)

,Model,Mean MAE,Std MAE,Mean RMSE,Std RMSE
0,Linear Regression,2.034322,0.095611,2.989281,0.427296
1,Ridge Regression,2.034609,0.094634,2.989229,0.427047
2,Decision Tree Regression,3.144527,0.340650,4.615152,0.562110


**The IQR check flagged 8 unusually long resolution times above about **16.43 hours**. They may be valid long-running tickets, not errors.compared the models using MAE and RMSE. MAE measures average prediction error, while RMSE gives larger errors more weight. Both metrics showed Linear Regression and Ridge performed almost equally and better than Decision Tree Regression.**

## REPORT

# Ticket Escalation and Resolution-Time Prediction

**Problem framing:** The goal is to predict whether a support ticket will be escalated and estimate its resolution time, helping support teams identify high-risk tickets and plan their workload.

## Introduction and Preprocessing

I removed `ticket_id` because it is only an identifier, not a useful predictive feature. I split the data into 75% training and 25% testing. For classification, stratification preserved the proportion of escalated and non-escalated tickets in both sets.

Numerical missing values were filled using the median and scaled, while categorical missing values were filled using the most frequent value and one-hot encoded. I placed preprocessing inside pipelines so each step was learned only from the training portion of each CV fold, preventing data leakage into validation data.

## Classification

I compared Logistic Regression, KNN, and Decision Tree using 5-fold StratifiedKFold and F1, since accuracy can be misleading with imbalanced classes. Stratification maintained class proportions across folds, making evaluation more reliable.

| Model | Mean F1 | Std. |
|---|---:|---:|
| Decision Tree | 0.4014 | 0.1041 |
| Logistic Regression | 0.3822 | 0.0538 |
| KNN | 0.3094 | 0.0738 |

Decision Tree had the highest mean F1, so I selected it for hyperparameter tuning. I tested `max_depth` (None, 3, 5, 7, 10, 15, 20) and `min_samples_split` (2, 5, 10). The best hyperparameters were max_depth=10 and min_samples_split=2, giving a CV F1 of **0.4267**. Standard deviation shows how much performance changes across folds; a lower value means more consistent results. Its standard deviation was higher than Logistic Regression's, showing less consistent performance across folds.

Test F1 was **0.3469**. The confusion matrix showed 144 true negatives, 33 false positives, 31 false negatives, and 17 true positives. The model missed 31 escalations, showing an important limitation.

## Regression

I compared Linear Regression, Ridge, and Decision Tree using 5-fold KFold and MAE. KFold was suitable because the target is continuous.

| Model | Mean MAE | Std. |
|---|---:|---:|
| Linear Regression | 2.0343 | 0.0956 |
| Ridge | 2.0346 | 0.0946 |
| Decision Tree | 3.1445 | 0.3407 |

Linear Regression had the lowest MAE, while Ridge was nearly tied and slightly more consistent. RMSE supported this comparison. An IQR check found 8 potential high outliers above 16.43 hours, which I retained because they may be genuine cases.

I tuned Linear Regression by testing the `fit_intercept=True/False` hyperparameter with values True and False. The best value was `True`, giving a CV MAE of 2.0343 close to the CV performance.

## Conclusion

I selected Decision Tree for classification and Linear Regression for regression based on their CV results. The test set was evaluated only once, after selection and tuning, to provide an independent estimate of performance. Repeatedly checking it could influence model choices and make evaluation less reliable.

In deployment, the cost of missed escalations, prediction speed, computing resources, and explainability could change the preferred models.